# Gold labeling — manual review of silver labels

In [5]:
import pandas as pd
from datetime import datetime
import os
import re
import numpy as np

In [253]:
SILVER_CSV      = '../data\silver_labels\silver_labels_v1_2026_06_21.csv'
CLEAN_DATA_CSV = '../data/alliance/MDM_Population_cleaned_v5_2026_06_21.parquet'
GOLD_LABELS_CSV = '../data/gold_labels/gold_labels_v1.csv'
ANYMATCH_PREDS_CSV = '../data/alliance/anymatch_predictions_full_v3.csv'
DET_RULES_PREDS_CSV = '../data/alliance/deterministic_rules_predictions_full_v1.csv'
FINAL_GOLD_LABELS_CSV = '../data/gold_labels/final_gold_labels_v1_2026_07_05.csv'

In [3]:
# Read silver labels. get_silver_labels.ipynb wrote with the index, so drop any
# unnamed index column and force the PATID columns to string (ID dtype trap).
silver = pd.read_csv(SILVER_CSV)

In [6]:
records = pd.read_parquet(CLEAN_DATA_CSV)
records['BirthDT_clean'] = records['BirthDT_clean'].dt.strftime('%Y-%m-%d')
records['Phones_set'] = records['Phones_set'].apply(lambda x: list(x) if isinstance(x, np.ndarray) else x)

In [7]:
anymatch_preds = pd.read_csv(ANYMATCH_PREDS_CSV)
anymatch_preds['pred'] = anymatch_preds['pred'].astype(bool)
det_rules_pred = pd.read_csv(DET_RULES_PREDS_CSV)

In [8]:
pairs = silver.merge(records.rename({c: c + '_A' for c in records.columns}, axis=1), 
             how='left', on='PATID_A')
pairs = pairs.merge(records.rename({c: c + '_B' for c in records.columns}, axis=1), 
             how='left', on='PATID_B')

In [9]:
pairs = pairs.merge(det_rules_pred.drop(columns=['rule_id', 'rule_reason']),  how='left', on=['PATID_A', 'PATID_B'])
pairs = pairs.merge(anymatch_preds.rename({'pred': 'anymatch_pred'}, axis=1).drop(columns=['match_prob']),  how='left', on=['PATID_A', 'PATID_B'])

In [10]:
ALL_FIELDS = ['FirstNM_raw', 'LastNM_raw', 'MiddleNM_raw', 'SuffixNM_raw',
       'BirthDT_raw', 'SSN_raw', 'AddressLine1_raw', 'AddressLine2_raw',
       'CityNM_raw', 'ZipCD_raw', 'StateCD_raw', 'CountryNM',
       'PrimaryPhoneNBR_raw', 'Phone01NBR_raw', 'Phone02NBR_raw',
       'Phone03NBR_raw', 'Email_raw', 'SexAtBirthDSC_raw', 'FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'PrimaryPhoneNBR_clean', 'Phone01NBR_clean', 'Phone02NBR_clean',
       'Phone03NBR_clean', 'Email_clean', 'SexAtBirthDSC_clean',
       'full_name_tokens', 'full_name_compact', 'Phones_set', 'Address_normalized']

CLEAN_FIELDS = ['FirstNM_clean',
       'LastNM_clean', 'MiddleNM_clean', 'SuffixNM_clean', 'BirthDT_clean',
       'SSN_clean', 'last_4_SSN', 'AddressLine1_clean', 'AddressLine2_clean',
       'CityNM_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext', 'StateCD_clean',
       'Email_clean', 'SexAtBirthDSC_clean', 'Phones_set']

In [11]:
import numpy as np
def show_pairs(df, k=50, fields=CLEAN_FIELDS):
    rows = []
    pairs = df.head(k).reset_index(drop=True)
    for _, r in pairs.iterrows():
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_A') for f in fields}})
        rows.append({'silver_label': r['silver_label'], **{f:r.get(f'{f}_B') for f in fields}})
        rows.append({'silver_label':''})  # spacer

    disp = pd.DataFrame(rows)

    def is_missing(v):
        return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == '')

    # color matrix matching disp shape
    colors = pd.DataFrame('', index=disp.index, columns=disp.columns)
    for i, r in pairs.iterrows():
        a_idx, b_idx = i * 3, i * 3 + 1  # display rows for this pair
        c = '#00C853' if r['silver_label'] else '#D50000'
        style = f'background-color: {c}'
        colors.loc[a_idx, 'silver_label'] = style
        colors.loc[b_idx, 'silver_label'] = style
        for f in fields:
            a, b = r.get(f'{f}_A'), r.get(f'{f}_B')
            am, bm = is_missing(a), is_missing(b)
            if am and bm:
                c = '#d3d3d3'  # grey  - both missing
            elif am or bm:
                c = '#fff3a3'  # yellow - one missing
            elif isinstance(a, list):
                if len(set(a + b)) == len(a) + len(b):
                    c = '#f4a8a8'  # red    - different
                else:
                    c = '#a8e6a3'  # green  - equal
            elif isinstance(a, np.ndarray):
                if len(set(a.tolist() + b.tolist())) == len(a) + len(b):
                    c = '#f4a8a8'  # red    - different
                else:
                    c = '#a8e6a3'  # green  - equal
            elif str(a) == str(b):
                c = '#a8e6a3'  # green  - equal
            else:
                c = '#f4a8a8'  # red    - different
            style = f'background-color: {c}'
            colors.loc[a_idx, f] = style
            colors.loc[b_idx, f] = style
    
    disp = disp.fillna('')

    # strip the _clean suffix on both frames together
    rename_map = {c: c.replace('_clean', '') for c in disp.columns}
    disp = disp.rename(columns=rename_map)
    colors = colors.rename(columns=rename_map)

    return disp.style.apply(lambda _: colors, axis=None)

In [330]:
# show_pairs(pairs, k=5)

In [331]:
# subset = pairs[(pairs['SSN_clean_A']==pairs['SSN_clean_B'])&(pairs['silver_label']==False)]
# show_pairs(subset)

## Browser-based gold-labeling app

`labeler.py` runs a tiny local Flask app (127.0.0.1 only). You pass it a **subset** of
`pairs`; it opens a browser tab with a scrollable, color-coded grid. Each pair has
**Match / No match** (→ `gold_label`) and **Ambiguous / OK** (→ `ambiguous_pair`) buttons,
plus keyboard shortcuts (`↑/↓` or `j/k` to move, `1` match, `2` no-match, `3` toggle ambiguous).

Every click writes to a **sparse** `gold_labels.csv` — one row per pair you've *touched*,
keyed by `(PATID_A, PATID_B)`. That CSV is the source of truth and survives restarts; the
notebook reads it back with `load_labels` / `apply_labels`.

Requires `flask` (`pip install flask`). Keep `gold_labels.csv` out of git — it's PHI-derived.

In [12]:
# The app lives in labeler.py (same folder). Re-import freely while iterating.
import importlib, labeler
importlib.reload(labeler)
from labeler import launch_labeler, load_labels, apply_labels, label_summary, stop_labeler

In [13]:
# # --- synthetic test set (real data not loadable yet) -----------------------
# # Builds a fake `pairs`-shaped DataFrame with the same *_A / *_B columns,
# # PATID pairs, silver_label, and a Phones_set list column, so we can exercise
# # every cell color (equal / different / one-missing / both-missing / list).
# import random
# _rng = random.Random(7)

# _firsts = ['John', 'Jon', 'Maria', 'María', 'Robert', 'Bob', 'Aisha', 'Wei', 'José', 'Jose']
# _lasts  = ['Smith', 'Smyth', 'Garcia', 'García', 'Nguyen', "O'Brien", 'Brown', 'Lee']
# _cities = ['Chicago', 'Cicero', 'Evanston', 'Berwyn']
# _streets = ['123 Main St', '45 Oak Ave', '900 Halsted St', '12 Lake Shore Dr']

# def _mk_record(seed_first, seed_last):
#     ssn = f'{_rng.randint(100,899)}-{_rng.randint(10,99)}-{_rng.randint(1000,9999)}'
#     return {'FirstNM_clean': seed_first, 'LastNM_clean': seed_last,
#             'MiddleNM_clean': _rng.choice([None, 'A', 'Marie', 'Lee']),
#             'SuffixNM_clean': _rng.choice([None, None, 'JR']),
#             'BirthDT_clean': f'19{_rng.randint(60,99)}-{_rng.randint(1,12):02d}-{_rng.randint(1,28):02d}',
#             'SSN_clean': ssn, 'last_4_SSN': ssn[-4:],
#             'AddressLine1_clean': _rng.choice(_streets), 'AddressLine2_clean': _rng.choice([None, 'Apt 2']),
#             'CityNM_clean': _rng.choice(_cities), 'ZipCD_clean_base': f'606{_rng.randint(10,99)}',
#             'ZipCD_clean_ext': _rng.choice([None, '1234']), 'StateCD_clean': 'IL',
#             'Email_clean': _rng.choice([None, f'{seed_first.lower()}@mail.com']),
#             'SexAtBirthDSC_clean': _rng.choice(['M', 'F']),
#             'Phones_set': [f'773555{_rng.randint(1000,9999)}' for _ in range(_rng.randint(1, 2))]}

# def _drop_some(rec, p=0.25):
#     rec = dict(rec)
#     for k in list(rec):
#         if k not in ('SSN_clean', 'last_4_SSN') and _rng.random() < p:
#             rec[k] = None
#     return rec

# def make_test_pairs(n=120):
#     rows = []
#     for i in range(n):
#         kind = _rng.choice(['exact', 'variant', 'ssn_diff', 'nomatch', 'missing'])
#         a = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts))
#         if kind == 'exact':
#             b, silver = dict(a), True
#         elif kind == 'variant':                      # same person, messy entry
#             b = dict(a); b['FirstNM_clean'] = a['FirstNM_clean'][:3]; b['AddressLine1_clean'] = _rng.choice(_streets)
#             silver = True
#         elif kind == 'ssn_diff':                     # same SSN, different people
#             b = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts))
#             b['SSN_clean'], b['last_4_SSN'] = a['SSN_clean'], a['last_4_SSN']
#             silver = False
#         elif kind == 'missing':                      # same person, lots of gaps
#             b = _drop_some(a, 0.5); silver = True
#         else:                                         # unrelated people
#             b = _mk_record(_rng.choice(_firsts), _rng.choice(_lasts)); silver = False
#         row = {'PATID_A': f'A{i:05d}', 'PATID_B': f'B{i:05d}',
#                'silver_label': silver, 'match_prob': round(_rng.random(), 4), 'scenario': kind}
#         row.update({f'{k}_A': v for k, v in a.items()})
#         row.update({f'{k}_B': v for k, v in b.items()})
#         rows.append(row)
#     return pd.DataFrame(rows)

# test_pairs = make_test_pairs(120)
# test_pairs[['PATID_A', 'PATID_B', 'silver_label', 'scenario']].head()

**Field comparison helpers**

In [14]:

# pandas treats NaN != NaN, so these compare only when BOTH sides are present:
#   _equal(col)   -> both present AND equal      (a missing side never counts as a match)
#   _differ(col)  -> both present AND different   (a missing side never counts as a diff)

def _present(col):
    return pairs[f'{col}_A'].notna() & pairs[f'{col}_B'].notna()

def _equal(col):
    return _present(col) & (pairs[f'{col}_A'].astype('string') == pairs[f'{col}_B'].astype('string'))

def _differ(col):
    return _present(col) & (pairs[f'{col}_A'].astype('string') != pairs[f'{col}_B'].astype('string'))

def _differ_or_missing(col):
    return pairs[f'{col}_A'].astype('string') != pairs[f'{col}_B'].astype('string')

def _phone_overlap():
    a, b = pairs['Phones_set_A'], pairs['Phones_set_B']
    return pd.Series(
        [bool(set(x) & set(y)) if isinstance(x, (list, set)) and isinstance(y, (list, set)) else False
         for x, y in zip(a, b)],
        index=pairs.index)

def _no_strong_match():
    # True when not one strong identifier agrees: SSN, last-4 SSN, DOB, email, phone, full name
    return ~(_equal('SSN_clean') | _equal('last_4_SSN') | _equal('BirthDT_clean')
             | _equal('Email_clean') | _phone_overlap() | _equal('full_name_compact'))


In [15]:
def add_gold_label_pairs(subset, is_match, is_ambiguous, gold_labels_path=GOLD_LABELS_CSV):
    is_match = 'match' if is_match else 'no_match'
    df_gold = pd.read_csv(gold_labels_path)
    df_gold.to_csv(os.path.dirname(gold_labels_path) + '/' + os.path.basename(gold_labels_path).replace('.csv', f'_backup_{datetime.now().strftime("%Y-%m-%d")}.csv'), index=False)
    
    gold_subset = subset[['PATID_A', 'PATID_B']].reset_index(drop=True)
    gold_subset['gold_label'] = is_match
    gold_subset['ambiguous_pair'] = is_ambiguous
    gold_subset['reviewed_at'] = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    
    # Drop existing rows whose (PATID_A, PATID_B) pair appears in the new subset
    pair_index = set(zip(gold_subset['PATID_A'], gold_subset['PATID_B']))
    mask = [pair not in pair_index for pair in zip(df_gold['PATID_A'], df_gold['PATID_B'])]
    df_gold = df_gold[mask]
    
    df_gold = pd.concat([df_gold, gold_subset], axis=0, ignore_index=True)
    df_gold.to_csv(gold_labels_path, index=False)

In [16]:
def add_indv_gold_label(patid_a, patid_b, is_match, is_ambiguous, gold_labels_path=GOLD_LABELS_CSV):
    is_match = 'match' if is_match else 'no_match'
    df_gold = pd.read_csv(gold_labels_path)
    mask = (df_gold['PATID_A'] == patid_a) & (df_gold['PATID_B'] == patid_b)
    if mask.sum() > 0:
        df_gold.loc[mask, ['gold_label', 'ambiguous_pair', 'reviewed_at']] = [is_match, is_ambiguous, datetime.now().strftime("%Y-%m-%dT%H:%M:%S")]
    else:
        new_row = pd.DataFrame({'PATID_A': [patid_a], 
                                'PATID_B': [patid_b], 
                                'gold_label': [is_match], 
                                'ambiguous_pair': [is_ambiguous], 
                                'reviewed_at': [datetime.now().strftime("%Y-%m-%dT%H:%M:%S")]})
        df_gold = pd.concat([df_gold, new_row], axis=0, ignore_index=True)
    
    df_gold.to_csv(gold_labels_path, index=False)

# Add current gold labels to pairs

In [231]:
pairs_gold = apply_labels(pairs, GOLD_LABELS_CSV)

# Manual Review

In [57]:
# Same SSN but not a match silver label
subset = pairs[(pairs.SSN_clean_A == pairs.SSN_clean_B) & (pairs.silver_label == False)]
subset.shape[0]

3

In [25]:
# same email and DOB but silver says no-match
subset = pairs[(_equal('Email_clean')) &
                (_equal('BirthDT_clean')) &
                (pairs.silver_label==False) ]
subset.shape[0]

33

Many cases marked as ambiguous due to possible female last name change

In [110]:
# same full name and DOB but silver says no-match AND SAME ADDRESS
subset = pairs[(_equal('full_name_compact'))&
                (_equal('BirthDT_clean'))&
                (_equal('AddressLine1_clean'))&
                (pairs.silver_label==False)]
subset.shape[0]

526

In [111]:
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [119]:
# One records has suffix JR, the other doesn't - > Marked as ambiguous
add_indv_gold_label(patid_a='66E2D752E5E9E01344DD690C50909C62', patid_b='8273523D3ABAEB4BBAA0882268309D98', is_match=True, is_ambiguous=True)

In [129]:
# Different SSN(typo) -> Not marked as ambiguous
# add_indv_gold_label(patid_a='3D666B5D42A8BFE71A8129CDD6CBDB9F', patid_b='CF9C8D5162CB2D193A2FA10B7D8CE08F', is_match=True, is_ambiguous=False)
# add_indv_gold_label(patid_a='743524FF088DECC5097226C178EB513F', patid_b='EEFE5C55CFD9A95BE2B45AA020C47C6B', is_match=True, is_ambiguous=False)

In [119]:
# Different sex -> Not marked as ambiguous

If we want to send cases where there is a typo in SSN or different sex at birth in the to_review section, we can do it programatically. These cases are clear matches, not ambiguous.

In [134]:
# different SSN but silver says match
subset = pairs[(_differ('SSN_clean'))&(pairs.silver_label==True)]
subset.shape[0]

307

Many had a lot typos in SSN -> Marked as a match non abiguous

Some had completely different SSN but same address and phone: Marked as a match but ambiguous

In [150]:
# different DOB but silver says match
# subset = pairs[(_differ('BirthDT_clean'))&(pairs.silver_label==True)]
subset = pairs[(_differ('BirthDT_clean'))&(pairs.silver_label==True)]
subset.shape[0]

103

In [151]:
# nothing strong agrees but silver says match
subset = pairs[(_no_strong_match())&(pairs.silver_label==True)]
subset.shape[0]

36

In [162]:
# same first, middle(populated) and last name and DOB but silver says no-match
subset = pairs[(_equal('FirstNM_clean'))&
    (_equal('LastNM_clean'))&
    (_equal('MiddleNM_clean'))&
    (_equal('BirthDT_clean'))&
    (pairs.silver_label==False)]
subset.shape[0]

173

In [163]:
# All of these cases are a match
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [194]:
# check if numbers in address are the same
# Same first name, last name, DOB and same initial numbers in address
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
subset = tmp_pairs[(_equal('FirstNM_clean'))&
    (_equal('LastNM_clean'))&
    (_equal('BirthDT_clean'))&
    (pairs_gold.silver_label==False) &
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
    (tmp_pairs['gold_label'].isna())]
subset.shape[0]

1136

In [195]:
# All of these cases are a match
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [217]:
# Same first name, last name, DOB and different initial numbers in address
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
subset = tmp_pairs[(_equal('FirstNM_clean'))&
    (_equal('LastNM_clean'))&
    (tmp_pairs['MiddleNM_clean_A'].notna()) &
    (tmp_pairs['MiddleNM_clean_B'].notna()) &
    (tmp_pairs['MiddleNM_clean_A']!=tmp_pairs['MiddleNM_clean_B']) &
    (_equal('BirthDT_clean'))&
    (pairs_gold.silver_label==False) &
    # (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']!=tmp_pairs['address_numbers_B']) &
    (tmp_pairs['gold_label'].isna())]
subset.shape[0]

111

Most of these the middle name doesn't agree because one record has the full middle name and the other just the initial. Manually annotated

In [301]:
# Same DOB, First, Last and suffix but silver label is false
subset= pairs[_equal('FirstNM_clean') &
    _equal('LastNM_clean') &
    _equal('BirthDT_clean') &
    (_present('SuffixNM_clean')) &
    (pairs['silver_label'] == False)]
subset.shape[0]

10

Marked all as a match

In [369]:
# same SSN but different DOB
subset = pairs_gold[(_equal('SSN_clean'))&(_differ('BirthDT_clean'))&(pairs_gold['gold_label'].isna())]
subset.shape[0]

0

Tehy were already labeled

In [375]:
# same SSN but different name and negative silver label
subset = pairs_gold[(_equal('SSN_clean'))&(_differ('full_name_compact'))&(pairs_gold['silver_label']==False)&(pairs_gold['gold_label'].isna())]
subset.shape[0]

0

In [376]:
# same SSN but different name and different DOB
subset = pairs_gold[(_equal('SSN_clean'))&(_differ('full_name_compact'))&(_differ('BirthDT_clean'))&(pairs_gold['gold_label'].isna())]
subset.shape[0]

0

In [378]:
# same SSN but different DOB
subset = pairs_gold[(_equal('SSN_clean'))&(_differ('BirthDT_clean'))&(pairs_gold['gold_label'].isna())]
subset.shape[0]

0

In [381]:
# same DOB and last name but different SSN (twins?)
subset = pairs[(_equal('BirthDT_clean'))&(_equal('LastNM_clean'))&(_differ('SSN_clean'))&(pairs_gold['gold_label'].isna())]
subset.shape[0]

86

Manually reviewed

In [386]:
# same name and DOB but different sex
subset = pairs[(_equal('full_name_compact'))&(_equal('BirthDT_clean'))&(_differ('SexAtBirthDSC_clean'))&(pairs_gold['gold_label'].isna())]
subset.shape[0]

732

No action taken

In [398]:
# Match on First name, last name, DOB  and same initial numbers in address(but different address)
# no match or both missing on address, email and phone(present or not)
# Middle name, suffix, SSN not present in at least one field
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
subset = tmp_pairs[_equal('FirstNM_clean') &
    _equal('LastNM_clean') &
    _equal('BirthDT_clean') &
    (~_equal('AddressLine1_clean')) &
    (~_equal('Email_clean')) &
    (~_phone_overlap()) &
    (~_present('SSN_clean')) &
    (~_present('MiddleNM_clean'))&
    (~_present('SuffixNM_clean')) &
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
    (tmp_pairs['gold_label'].isna())]
subset.shape[0]

2285

In [400]:
# All marked as match not ambiguous -> It would be a good deterministic rule
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [405]:
# Match on First name, last name, DOB
# no match or both missing on address, email and phone(present or not)
# Middle name, suffix, SSN not present in at least one field
subset= pairs[_equal('FirstNM_clean') &
    _equal('LastNM_clean') &
    _equal('BirthDT_clean') &
    (~_equal('AddressLine1_clean')) &
    (~_equal('Email_clean')) &
    (~_phone_overlap()) &
    (~_present('SSN_clean')) &
    (~_present('MiddleNM_clean'))&
    (~_present('SuffixNM_clean'))  &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

20717

In [409]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [421]:
# same last name, dob and address but different first name 
subset = pairs[(_equal('LastNM_clean'))&(_equal('AddressLine1_clean'))&(_differ('FirstNM_clean'))&
    (_equal('BirthDT_clean')) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

1230

Manually reviewed

In [435]:
# Same first, last and DOB but gold or silver labels says not a match
subset = pairs_gold[_equal('FirstNM_clean')&_equal('LastNM_clean') &_equal('BirthDT_clean') & (pairs_gold['silver_label']==False) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

0

In [462]:
# Same First name, DOB, phone and first number in address and silver label says it is not a match

tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')

subset = pairs_gold[_equal('FirstNM_clean')&_equal('BirthDT_clean') & (pairs_gold['silver_label']==False) & (_phone_overlap()) &
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

152

In [464]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [465]:
# Same First name, DOB, phone and first number in address and silver label says it is not a match

tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')

subset = pairs_gold[_equal('FirstNM_clean')&_equal('BirthDT_clean') & (pairs_gold['silver_label']==False) & (_phone_overlap()) &
    (tmp_pairs['address_numbers_A']!=tmp_pairs['address_numbers_B']) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

385

In [467]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [473]:
# Same First name, DOB, and first number in address and silver label says it is not a match
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')

subset = pairs_gold[_equal('FirstNM_clean')&_equal('BirthDT_clean') & (pairs_gold['silver_label']==False) &
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

207

In [475]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [479]:
# Same First name, DOB and silver label says it is not a match ( The rest not marked already in the gold labels set)
subset = pairs_gold[_equal('FirstNM_clean')&_equal('BirthDT_clean') & (pairs_gold['silver_label']==False) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

841

In [480]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

Manuall review over these changing to not ambiguous match or not a match for certain clear cases

In [484]:
# same full name and DOB but silver says no-match
subset = pairs_gold[(_equal('full_name_compact'))&(_equal('BirthDT_clean'))&(pairs_gold.silver_label==False)&(pairs_gold['gold_label'].isna())]
subset.shape[0]

131

In [486]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [492]:
# Same phone number but not a match
subset = pairs_gold[(pairs_gold['silver_label']==False) & (_phone_overlap()) & (_equal('BirthDT_clean'))&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

169

Manually reviewed

In [18]:
# same name and address but silver labels says not a match
subset = pairs_gold[(pairs_gold['silver_label']==False) & _equal('full_name_compact')& _equal('AddressLine1_clean')&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

14

Manually reviewed

In [46]:
# Woman that changes her last name or typo in the last name with same first name, DOB and phone overlap and different (or missing) address and email
subset = pairs_gold[_equal('FirstNM_clean')&(_differ('LastNM_clean'))& (_phone_overlap())&(_equal('BirthDT_clean'))&
    _differ_or_missing('AddressLine1_clean')&
    _differ_or_missing('Email_clean')&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

298

Manually reviewed

In [59]:
# Same first name, last name, phone and DOB
subset = pairs_gold[_equal('FirstNM_clean')&(_equal('LastNM_clean'))& (_phone_overlap())&(_equal('BirthDT_clean'))&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

15302

In [60]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [78]:
# Same first name, phone, first number in address and DOB, different last name. Anymathc says it is a match deterministic ruels say it isn't
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')

subset = pairs_gold[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')&
    _equal('FirstNM_clean')&(_differ('LastNM_clean'))& (_phone_overlap())&(_equal('BirthDT_clean'))&
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

509

In [80]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [89]:
# Same first name, phone, email and DOB, different last name. Anymathc says it is a match deterministic ruels say it isn't
subset = pairs_gold[ _equal('FirstNM_clean')&(_differ('LastNM_clean'))& (_phone_overlap())&(_equal('BirthDT_clean'))&_equal('Email_clean')&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

535

In [90]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [93]:
# Same first name, phone and DOB, different last name. Anymathc says it is a match deterministic ruels say it isn't. Rest of records after labeling the previous
subset = pairs_gold[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')&
    _equal('FirstNM_clean')&(_differ('LastNM_clean'))& (_phone_overlap())&(_equal('BirthDT_clean'))&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

503

In [97]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [105]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says a match, det rules says it is not a match. Same email
subset = pairs[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')&(_equal('Email_clean'))&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

97

Manually reviewed

In [120]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says a match, det rules says it is not a match. Differ first and last name
subset = pairs[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')&(_differ('FirstNM_clean'))&(_differ('LastNM_clean'))&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

68

Manually reviewed

In [134]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says a match, det rules says it is not a match. Same address
subset = pairs[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')& _equal('AddressLine1_clean')&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

13

In [135]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [146]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says a match, det rules says it is not a match. Different phone
subset = pairs[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')& (~_phone_overlap())& 
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

383

In [150]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [152]:
# Where Deterministic rules and anymatch didn't agree
# Anymatch says a match, det rules says it is not a match
subset = pairs[(pairs['anymatch_pred'] == True)&(pairs['rule_pred']=='non_match')&
    (pairs_gold['gold_label'].isna())]
subset.shape[0]

131

Manually reviewed

In [158]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. different email, phone, address

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (~_phone_overlap())& _differ('Email_clean')
    &_differ('AddressLine1_clean')&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

2

Manually reviewed

In [171]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. different phone, equal email

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (~_phone_overlap())&
    _equal('Email_clean')&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

15

Manually reviewed

In [184]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. Equal SSN and diff first name

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (_equal('SSN_clean'))& _differ('FirstNM_clean')&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

26

Manually reviewed

In [187]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. Equal SSN 

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (_equal('SSN_clean'))&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

275

In [189]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [202]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. Same first numbers in address
tmp_pairs = pairs_gold.copy()
tmp_pairs['address_numbers_A'] = tmp_pairs['AddressLine1_clean_A'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')
tmp_pairs['address_numbers_B'] = tmp_pairs['AddressLine1_clean_B'].apply(lambda x: re.findall('^[0-9]+', x)[0] if isinstance(x, str) and re.findall('^[0-9]+', x) else '')

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & 
    (tmp_pairs['AddressLine1_clean_A'].notna()) &
    (tmp_pairs['address_numbers_A']==tmp_pairs['address_numbers_B']) &
                pairs_gold['gold_label'].isna()]
subset.shape[0]

161

In [203]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [207]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. Equal SSN 

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (_phone_overlap())&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

290

Manually reviewed

In [222]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. Equal SSN 
tmp_pairs = pairs.copy()
tmp_pairs['MiddleNM_initial_A'] = tmp_pairs['MiddleNM_clean_A'].str[0]
tmp_pairs['MiddleNM_initial_B'] = tmp_pairs['MiddleNM_clean_B'].str[0]
subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (~_phone_overlap())&
    (tmp_pairs['MiddleNM_initial_A']==tmp_pairs['MiddleNM_initial_B'])&
                pairs_gold['gold_label'].isna()]
subset.shape[0]

25

In [224]:
# All marked as match and not ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=False)

In [229]:
# Where Deterministic rules and anymatch didn't agree. Anymatch says not a match, det rules says it is a match. 
# The rest only matches on first name, last name(both with typos) and same DOB

subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & 
                pairs_gold['gold_label'].isna()]
subset.shape[0]

421

In [230]:
# All marked as match AND ambiguous
add_gold_label_pairs(subset, is_match=True, is_ambiguous=True)

In [266]:
# subset = pairs_gold[(pairs_gold['anymatch_pred'] == False)&(pairs_gold['rule_pred']=='match') & (_equal('SSN_clean'))& _differ('FirstNM_clean')]
# subset.shape[0]

32

In [23]:
CLEAN_FIELDS

['FirstNM_clean',
 'LastNM_clean',
 'MiddleNM_clean',
 'SuffixNM_clean',
 'BirthDT_clean',
 'SSN_clean',
 'last_4_SSN',
 'AddressLine1_clean',
 'AddressLine2_clean',
 'CityNM_clean',
 'ZipCD_clean_base',
 'ZipCD_clean_ext',
 'StateCD_clean',
 'Email_clean',
 'SexAtBirthDSC_clean',
 'Phones_set']

In [ ]:
show_pairs(subset, k=100)

# TODO Review list

In [ ]:

# Identify changes of last name. When we should mark it as ambiguous?


# Same name, DOB, different last name, same phone, missing everything else. Should it be a match and ambiguous or not?

# Observations

- The number of typos in SSN is huge. The typos are clearly confusions of handwriting. For example confusing 3 and 8, 1 and 7, etc.
- There are typos in DOB too, sometimes switching month and day
- Many typos in names
- There are a lot of cases of what looks to be the same person(uncommon name) but with different SSN and it is not a typo
- Many cases where it seems to be the same person bu sex is different
- One very interesting identifier is the first numbers of the address. As the full address name disagrees often.
- Hard case that I have to manually review. A lot of things in common but different first anme. Sometimes is misspelling, sometimes it is abbreviations(Nick vs Nickolas) others are different people. Silver labels make many mistakes here.
- There are so many first name mispellings
- Hardest cases are those that have same last anem, DOB, address and different first name. Many FP and FN
- Many that share First name, DOB and there is a typo or mispelled in last name were wrongly classified as not a match. Although many others are classified correctly as a match.
- If they share only first name, last name and DOB I marked it as a match but ambiguous

# Launch labeler

In [267]:
launch_labeler(subset, store_path=GOLD_LABELS_CSV)

[labeler] running at http://127.0.0.1:8777/
[labeler] store: C:\Users\MiguelGarcia\Documents\AnyMatch\data\gold_labels\gold_labels_v1.csv  (32 pairs loaded)
[labeler] call stop_labeler(8777) to shut it down


'http://127.0.0.1:8777/'

127.0.0.1 - - [06/Jul/2026 10:03:54] "GET / HTTP/1.1" 200 -


In [234]:
stop_labeler(8765)

In [260]:
pairs_gold = apply_labels(pairs, GOLD_LABELS_CSV)

In [248]:
pairs_gold['final_gold_label'] = pairs_gold.apply(lambda row: row['gold_label'] 
                                                  if row['gold_label'] is not np.nan else ('match' if row['silver_label']==True else 'no_match'), axis=1)

In [261]:
pairs_gold['final_gold_label'] = pairs_gold.apply(lambda row: True if row['gold_label'] == 'match' else 
                                                  (False if row['gold_label'] == 'no_match' else row['silver_label']==True), axis=1)

In [262]:
pairs_gold[['silver_label', 'gold_label', 'ambiguous_pair', 'final_gold_label']]

,silver_label,gold_label,ambiguous_pair,final_gold_label
0,True,match,False,True
1,True,match,False,True
2,True,match,False,True
3,True,match,False,True
4,False,match,True,True
...,...,...,...,...
204800,False,NaN,False,False
204801,False,NaN,False,False
204802,True,NaN,False,True
204803,True,NaN,False,True


In [263]:
pairs_gold[['PATID_A', 'PATID_B', 'ambiguous_pair', 'final_gold_label']].to_csv(FINAL_GOLD_LABELS_CSV, index=False)